<a href="https://colab.research.google.com/github/nyloanon/bachelor_thesis/blob/main/khi_random_init.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2D Kelvin Helmholtz Instability

## Imports

In [ ]:
!pip install astronomix -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.9/172.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.5 MB/s eta 0:00:00


In [ ]:
# numerics
import jax
import jax.numpy as jnp
from jax.random import PRNGKey, uniform

# timing
from timeit import default_timer as timer

# plotting
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm
import matplotlib.animation as animation

# astronomix
from astronomix import SimulationConfig
from astronomix import get_helper_data
from astronomix import SimulationParams
from astronomix import time_integration
from astronomix.option_classes.simulation_config import SnapshotSettings
from astronomix import construct_primitive_state
from astronomix import get_registered_variables
from astronomix.option_classes.simulation_config import finalize_config
from astronomix.option_classes.simulation_config import (
    BACKWARDS,
    DOUBLE_MINMOD,
    FORWARDS,
    HLL,
    HLLC,
    HYBRID_HLLC,
    MINMOD,
    OSHER,
    PERIODIC_BOUNDARY,
    BoundarySettings,
    BoundarySettings1D,
    FINITE_VOLUME,
    FINITE_DIFFERENCE
)

## Colab print fix

Otherwise the progress bar does not work in colab. I've not faced any issues in Jupyter Notebooks in VSCode.

In [ ]:
import builtins
from IPython.display import clear_output

if not hasattr(builtins, "_safe_original_print"):
    builtins._safe_original_print = builtins.print

def _colab_compatible_print(*args, **kwargs):
    """
    Intertcepts print calls. If it detects a progress bar (starts with \r),
    it uses Colab's clear_output instead of carriage returns.
    """
    # Check if the text starts with '\r' (which your function does)
    if args and isinstance(args[0], str) and args[0].startswith('\r'):
        # 1. Clear the output area to prevents "waterfall" effect
        clear_output(wait=True)

        # 2. Strip '\r' and trailing spaces to prevent line wrapping
        clean_text = args[0].replace('\r', '').rstrip()

        # 3. Print using the SAFE original function
        builtins._safe_original_print(clean_text)
    else:
        # For all other print calls (including the final newline), use the SAFE original
        builtins._safe_original_print(*args, **kwargs)

# Overwrite print with our wrapper
builtins.print = _colab_compatible_print

## Basic simulation setup

In [ ]:
print("👷 Setting up simulation...")

# simulation settings
gamma = 5/3

# spatial domain
box_size = 1.0
num_cells = 128

fixed_timestep = False
scale_time = False
dt_max = 0.1
num_timesteps = 2000

# setup simulation config
config = SimulationConfig(
    progress_bar = False,
    dimensionality = 2,
    box_size = box_size,
    num_cells = num_cells,
    fixed_timestep = fixed_timestep,
    differentiation_mode = FORWARDS,
    num_timesteps = num_timesteps,
    boundary_settings = BoundarySettings(
        x = BoundarySettings1D(PERIODIC_BOUNDARY, PERIODIC_BOUNDARY),
        y = BoundarySettings1D(PERIODIC_BOUNDARY, PERIODIC_BOUNDARY)
    ),
    limiter = DOUBLE_MINMOD,
    return_snapshots = False,
    riemann_solver = HYBRID_HLLC,
    solver_mode = FINITE_VOLUME
)

helper_data = get_helper_data(config)

params = SimulationParams(
    t_end = 2.0,
    C_cfl = 0.4
)

registered_variables = get_registered_variables(config)

👷 Setting up simulation...


## Plotting helper

In [ ]:
def produce_plot(final_state, index):
  s = 0.1

  fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

  # equal aspect ratio
  ax1.set_aspect('equal', 'box')
  ax2.set_aspect('equal', 'box')
  ax3.set_aspect('equal', 'box')

  x = jnp.linspace(0, box_size, num_cells)
  y = jnp.linspace(0, box_size, num_cells)

  ym, xm = jnp.meshgrid(x, y)

  # on the first axis plot the density
  # log scaler
  norm_rho = LogNorm(vmin = jnp.min(final_state[0, :, :]), vmax = jnp.max(final_state[0, :, :]), clip = True)
  norm_p = LogNorm(vmin = jnp.min(final_state[3, :, :]), vmax = jnp.max(final_state[3, :, :]), clip = True)

  # ax1.scatter(xm.flatten(), ym.flatten(), c = final_state[0, :, :].flatten(), s = s, norm = norm_rho, marker = "s", cmap = "jet")
  # ax1.set_title("Density")

  ax1.imshow(final_state[0, :, :].T, norm = norm_rho, cmap = "jet", origin = "lower", extent = [0, box_size, 0, box_size])
  ax1.set_title("Density")

  # on the second axis plot the absolute velocity
  # abs_vel = jnp.sqrt(final_state[1, :, :]**2 + final_state[2, :, :]**2)

  # vel_norm = LogNorm(vmin = jnp.min(abs_vel), vmax = jnp.max(abs_vel), clip = True)

  ax2.imshow(final_state[1, :, :].T, cmap = "jet", origin = "lower", extent = [0, box_size, 0, box_size])
  ax2.set_title("Velocity")

  # on the third axis plot the pressure
  ax3.imshow(final_state[4, :, :].T, norm = norm_p, cmap = "jet", origin = "lower", extent = [0, box_size, 0, box_size])
  ax3.set_title("Pressure")

  plt.savefig(f"final_state_{index}.png")

## KHI initialization

In [ ]:
def random_khi_fourier_modes(
    key,
    X,
    Y,
    amplitude=0.01,
    k_min=1,
    k_max=8,
    shear_layers=(0.25, 0.75),
    width=0.03,
    spectral_slope=1.0,
):
    """
    Random Fourier-mode perturbation for Kelvin-Helmholtz initialization.

    Produces a y-velocity perturbation of the form

        u_y(x, y) = A f(y) sum_k a_k sin(2 pi k x + phi_k)

    where f(y) localizes the perturbation around the shear layers.
    """

    modes = jnp.arange(k_min, k_max + 1)
    num_modes = modes.shape[0]

    key_amp, key_phase = jax.random.split(key)

    # random amplitudes with optional spectral decay
    coeffs = jax.random.normal(key_amp, (num_modes,))
    coeffs = coeffs / modes**spectral_slope

    # normalize so amplitude is controlled by `amplitude`
    coeffs = coeffs / jnp.sqrt(jnp.sum(coeffs**2) + 1e-30)

    phases = jax.random.uniform(
        key_phase,
        (num_modes,),
        minval=0.0,
        maxval=2.0 * jnp.pi,
    )

    # shape: (num_modes, nx, ny)
    fourier_sum = jnp.sum(
        coeffs[:, None, None]
        * jnp.sin(2.0 * jnp.pi * modes[:, None, None] * X[None, :, :] + phases[:, None, None]),
        axis=0,
    )

    # localize perturbation around both shear layers
    envelope = jnp.zeros_like(Y)
    for y0 in shear_layers:
        envelope = envelope + jnp.exp(-0.5 * ((Y - y0) / width) ** 2)

    return amplitude * envelope * fourier_sum

## Data generation

In [ ]:
# Grid size and configuration
num_cells = config.num_cells
x = jnp.linspace(0, 1, num_cells)
y = jnp.linspace(0, 1, num_cells)
X, Y = jnp.meshgrid(x, y, indexing="ij")

# Initialize state
rho = jnp.ones_like(X)
u_x = 0.5 * jnp.ones_like(X)

# deterministic setup
# u_y = 0.01 * jnp.sin(2 * jnp.pi * X)

# random initialization
key = PRNGKey(0)
num_sims = 2000

for i in range(num_sims):
    key, subkey = jax.random.split(key)

    # Base state
    rho = jnp.ones_like(X)
    u_x = 0.5 * jnp.ones_like(X)

    # Slab mask
    mask = (Y > 0.25) & (Y < 0.75)

    # between y = 0.25 and y = 0.75 set u_x to -0.5 and rho to 2.0
    u_x = jnp.where(mask, -0.5, u_x)
    rho = jnp.where(mask, 2.0, rho)

    # KHI-suited random Fourier perturbation
    u_y = random_khi_fourier_modes(
        subkey,
        X,
        Y,
        amplitude=0.01,
        k_min=1,
        k_max=8,
        shear_layers=(0.25, 0.75),
        width=0.03,
        spectral_slope=1.0,
    )

    # Initialize pressure
    p = jnp.ones((num_cells, num_cells)) * 2.5

    # Initial state
    initial_state = construct_primitive_state(
        config=config,
        registered_variables=registered_variables,
        density=rho,
        velocity_x=u_x,
        velocity_y=u_y,
        gas_pressure=p,
    )

    config = finalize_config(config, initial_state.shape)

    final_state = time_integration(
        initial_state,
        config,
        params,
        registered_variables,
    )

    produce_plot(final_state, i)

/tmp/ipykernel_2794/3170462946.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))
